# Simple RAG (Retrieval-Augmented Generation) — Beginner Walkthrough

RAG lets an LLM answer questions using **your own documents** instead of only what it memorized during training. This notebook shows every step, one at a time, with nothing hidden inside a library function.

```
  DOCUMENT              INDEXING (done once)                         QUERY TIME (every question)
┌───────────┐   split   ┌───────────┐  embed  ┌────────────┐        ┌───────────┐  embed  ┌────────────┐
│  raw text │ ───────▶ │  chunks   │ ──────▶ │  vector DB │        │ user query│ ──────▶ │ query vector│
└───────────┘           └───────────┘         └────────────┘        └───────────┘         └──────┬─────┘
                                                     │                                             │
                                                     │      semantic search (similarity)           │
                                                     └─────────────────────◀──────────────────────┘
                                                                     │
                                                          top-k relevant chunks
                                                                     │
                                                                     ▼
                                                     context + question ──▶ LLM ──▶ plain text answer
```

**The 7 steps we'll code:**
1. Take a document
2. Chunk it into small pieces
3. Embed each chunk into a vector
4. Store the vectors in a vector database
5. Embed the user's query the same way
6. Retrieve the most similar chunks (semantic search)
7. Give the retrieved chunks + question to the LLM → get a plain text answer

> **What kind of RAG is this?** This is the most basic form — often called **"Naive RAG"** in the literature. One retrieval pass, dense/semantic vector search, fixed top-k, no re-ranking, no agent loop. See the note at the bottom for how this compares to fancier variants (Advanced RAG, Agentic RAG, Graph RAG, etc.).


In [ ]:
!pip install -qU langchain langchain-groq langchain-community langchain-huggingface faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("Groq API key loaded!")

Groq API key loaded!


## Step 1: The document

This is our "knowledge base" — the private/custom text the LLM doesn't already know. In a real project this could be a PDF, a website, or a folder of company docs. Here we'll just use a plain Python string so the whole pipeline is easy to see.

Notice each paragraph covers **one distinct fact** — that makes it easy to see retrieval working later.


In [ ]:
document_text = """
BookNook was founded in 2015 as a small online bookstore specializing in independent and
self-published authors. Today we carry over 40,000 titles across fiction, non-fiction, and
children's books, and our mission is to help readers discover books they won't find on the
shelves of big chain stores.

Standard shipping takes 3 to 5 business days and is free on orders over $30. Orders under $30
have a flat shipping fee of $4.99. We also ship internationally to over 20 countries, with
delivery times of 7 to 14 business days depending on the destination.

We accept returns within 30 days of delivery, as long as the book is in its original condition.
Refunds are issued to the original payment method and typically appear within 5 to 7 business
days after we receive the returned item. Digital e-book purchases are non-refundable.

BookNook Plus is our paid membership program, priced at $39 per year. Members get free shipping
on every order regardless of amount, early access to pre-orders, and a permanent 10% discount
on all print books.

Our customer support team is available Monday through Saturday, 9:00 AM to 6:00 PM. You can
reach us by email at [email protected], through live chat on our website, or by phone at
1-800-555-0199.
"""

## Step 2: Chunk it

LLMs and embedding models work better on small, focused pieces of text rather than one huge blob. We split the document into overlapping chunks:
- `chunk_size` — max characters per chunk
- `chunk_overlap` — characters repeated between consecutive chunks, so we don't cut a sentence in half and lose meaning at the boundary


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)

chunks = splitter.create_documents([document_text])

print(f"Document was split into {len(chunks)} chunks.\n")
for i, c in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(c.page_content.strip())
    print()

Document was split into 5 chunks.

--- Chunk 0 ---
BookNook was founded in 2015 as a small online bookstore specializing in independent and
self-published authors. Today we carry over 40,000 titles across fiction, non-fiction, and
children's books, and our mission is to help readers discover books they won't find on the
shelves of big chain stores.

--- Chunk 1 ---
Standard shipping takes 3 to 5 business days and is free on orders over $30. Orders under $30
have a flat shipping fee of $4.99. We also ship internationally to over 20 countries, with
delivery times of 7 to 14 business days depending on the destination.

--- Chunk 2 ---
We accept returns within 30 days of delivery, as long as the book is in its original condition.
Refunds are issued to the original payment method and typically appear within 5 to 7 business
days after we receive the returned item. Digital e-book purchases are non-refundable.

--- Chunk 3 ---
BookNook Plus is our paid membership program, priced at $39 per yea

## Step 3: Embed each chunk

An embedding model turns text into a list of numbers (a vector) that captures its *meaning*. Text with similar meaning ends up with vectors that are close together, even if the wording is completely different.

We use a small, free, open-source embedding model from HuggingFace (`all-MiniLM-L6-v2`) — it runs locally, no extra API key needed.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Quick peek at what an embedding actually looks like
sample_vector = embedding_model.embed_query("How long does shipping take?")
print("Vector length:", len(sample_vector))
print("First 8 numbers:", sample_vector[:8])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector length: 384
First 8 numbers: [0.0018660066416487098, -0.00843244418501854, -0.009860513731837273, 0.075279101729393, -0.008166038431227207, -0.08819602429866791, -0.02510788105428219, -0.029266050085425377]


## Step 4: Store the embeddings in a vector database

A vector database stores each chunk's embedding alongside the original text, and is built to quickly find the *closest* vectors to any new vector we give it. We use **FAISS**, a simple in-memory vector store — perfect for learning, no server or account needed.


In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embedding_model)
print("Vector database built:", vectorstore.index.ntotal, "vectors stored.")

/tmp/ipykernel_2617/2724741861.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Vector database built: 5 vectors stored.


## Step 5 & 6: The retriever — embed the query, then semantic search

A **retriever** wraps the vector database. When we call it with a user's question, it:
1. embeds the question with the *same* embedding model (Step 5), then
2. finds the `k` chunks whose vectors are closest to the question's vector (Step 6 — this is the "semantic search").

Let's test the retriever **on its own first**, before involving the LLM at all, so you can see exactly what gets retrieved.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

test_results = retriever.invoke("How long does shipping take?")

print("Top matching chunks for the query:\n")
for i, doc in enumerate(test_results):
    print(f"Match {i+1}:")
    print(doc.page_content.strip())
    print()

Top matching chunks for the query:

Match 1:
Standard shipping takes 3 to 5 business days and is free on orders over $30. Orders under $30
have a flat shipping fee of $4.99. We also ship internationally to over 20 countries, with
delivery times of 7 to 14 business days depending on the destination.

Match 2:
We accept returns within 30 days of delivery, as long as the book is in its original condition.
Refunds are issued to the original payment method and typically appear within 5 to 7 business
days after we receive the returned item. Digital e-book purchases are non-refundable.



## Step 7: Give the retrieved chunks + question to the LLM

Now we build the full RAG chain:
`question → retriever → relevant chunks → stuffed into a prompt with the question → LLM → plain text answer`

The prompt explicitly tells the LLM to answer **only** using the retrieved context — this is what keeps RAG answers grounded instead of the model just making things up.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

rag_prompt = ChatPromptTemplate.from_template(
    "You are a helpful customer support assistant for the bookstore BookNook. Answer the "
    "question using ONLY the context below. If the answer is not in the context, say you "
    "don't know — do not make anything up.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

def format_docs(docs):
    return "\n\n".join(doc.page_content.strip() for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

## Try it out!

Notice the last question isn't covered anywhere in our document — a well-grounded RAG system should say it doesn't know, instead of guessing.


In [ ]:
questions = [
    "How long does shipping take?",
    "What's your return policy?",
    "What do I get with BookNook Plus?",
    "What are your customer support hours?",
    "What is the capital of France?",  # not in our document — tests grounding
]

for q in questions:
    answer = rag_chain.invoke(q)
    print("Q:", q)
    print("A:", answer)
    print("-" * 60)

Q: How long does shipping take?
A: Standard shipping takes 3 to 5 business days. For international orders, delivery typically takes 7 to 14 business days, depending on the destination.
------------------------------------------------------------
Q: What's your return policy?
A: We accept returns within 30 days of delivery, provided the book is in its original condition. Refunds are issued to the original payment method and usually appear within 5–7 business days after we receive the returned item. (Note that digital e‑book purchases are non‑refundable.)
------------------------------------------------------------
Q: What do I get with BookNook Plus?
A: With BookNook Plus you receive:

- Free shipping on every order, no matter the amount  
- Early access to pre‑orders  
- A permanent 10 % discount on all print books  

(The membership costs $39 per year.)
------------------------------------------------------------
Q: What are your customer support hours?
A: Our customer support team is

**Try it yourself:** ask your own questions interactively:

```python
while True:
    q = input("Ask a question (or 'quit'): ")
    if q.lower() == "quit":
        break
    print("A:", rag_chain.invoke(q))
```


## Summary

| Step | What happens | Code |
|---|---|---|
| 1. Document | Raw source text | `document_text` |
| 2. Chunking | Split into small overlapping pieces | `RecursiveCharacterTextSplitter` |
| 3. Embedding | Each chunk → a vector of numbers | `HuggingFaceEmbeddings` |
| 4. Vector DB | Store chunk vectors for fast similarity search | `FAISS.from_documents()` |
| 5–6. Retriever | Embed the query, find the closest chunk vectors (semantic search) | `vectorstore.as_retriever()` |
| 7. LLM | Answer the question using only the retrieved chunks | `rag_prompt \| llm \| StrOutputParser()` |

## What kind of RAG is this, exactly?

This notebook builds what's usually called **Naive RAG** (sometimes "Simple" or "Vanilla" RAG) — the baseline architecture every other RAG variant builds on top of. It has these specific properties:

- **Dense / semantic retrieval** — we search by embedding similarity, not keyword matching (that alternative is called *sparse retrieval*, e.g. BM25/TF-IDF).
- **Single-pass retrieval** — one retrieval call per question. No looping back to search again, no multi-hop reasoning.
- **Fixed top-k** — we always grab exactly `k=2` chunks, regardless of how relevant they actually are.
- **No query rewriting** — the user's raw question is embedded as-is, unlike techniques such as HyDE or multi-query expansion that rephrase the question first.
- **No re-ranking** — whatever the vector search returns goes straight into the prompt, with no second model scoring/reordering the results.
- **Stateless** — each question is independent; there's no conversation memory feeding into retrieval (that would be *Conversational RAG*).

For teaching, it's worth mentioning what students would add next if they kept going:

| Variant | What it adds over Naive RAG |
|---|---|
| **Advanced RAG** | Query rewriting, hybrid (dense+sparse) search, re-ranking retrieved chunks before generation |
| **Modular RAG** | Swappable/multiple retrieval modules, routing logic to pick which knowledge source to query |
| **Agentic RAG** | The LLM decides *whether*, *when*, and *how many times* to retrieve — can loop or do multi-hop retrieval |
| **Graph RAG** | Retrieves from a knowledge graph instead of (or alongside) a vector store, to capture relationships between facts |
| **Conversational RAG** | Chat history is folded into the retrieval query, so follow-up questions work |

### Things to try with students
- Change `k` in `as_retriever(search_kwargs={"k": 2})` — what happens with `k=1` vs `k=4`?
- Shrink `chunk_size` a lot (e.g. 50) — watch retrieval get worse because facts get split apart.
- Ask a question the document doesn't cover — show the model correctly saying "I don't know."
- Swap `document_text` for a completely different topic and re-run everything from Step 1.
